# 🔬 Lab 3 — Inside a GPU
**Foothold · GPU Computing for AI · Unit 3 · Book: Step 3**

Open up your GPU: count its parts, make its warps diverge, watch it hide memory latency, measure its bandwidth, and check your capstone plan.
For the first time you'll run small **CUDA C kernels**, the code that each GPU thread runs, through the CuPy library.

| Part | What you do | Goes with Foothold lesson |
|---|---|---|
| A | Compute the T4's peak, then measure FP32 vs FP16 | 3.1 SMs, Cores & Tensor Cores |
| B | Make a warp diverge and time the damage | 3.2 Warps & Branch Divergence |
| C | Add threads until memory latency is hidden | 3.3 Hiding Latency |
| D | Measure bandwidth: coalesced vs strided | 3.4 GPU Memory & Bandwidth |
| E | Your capstone plan vs reality | 3.C Capstone |

⏱️ About 90 minutes. 📘 Textbook: Kirk & Hwu Ch 4–5 · H&P §4.4 · Hamacher §12.1, Ch 8. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after `#` in Python, after `//` inside the CUDA C kernels).

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run both cells

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · Work Out the Peak, Then Measure  *(Unit 3.1)*

Peak FLOPS = SMs × units per SM × FLOPs per unit per clock × clock rate. For the T4: 40 SMs; 64 FP32 cores and 8 Tensor Cores per SM; one FMA (2 FLOPs) per core per clock; 64 FMAs per Tensor Core per clock.

In [ ]:
# the T4's published design: 40 SMs, each with 64 FP32 cores and 8 Tensor Cores, boost clock 1.59 GHz
t4_sms, clock_ghz = 40, 1.59
# FP32: SMs × 64 cores × 2 FLOPs per fused multiply-add × clocks per second
fp32_peak = t4_sms * 64 * 2 * clock_ghz * 1e9
# FP16 Tensor Cores: SMs × 8 Tensor Cores × 64 multiply-adds each × 2 FLOPs × clocks per second
fp16_peak = t4_sms * 8 * 64 * 2 * clock_ghz * 1e9
print(f"T4 peak FP32 (CUDA cores)  : {fp32_peak / 1e12:5.1f} TFLOPS")   # ÷ 1e12 gives TFLOPS
print(f"T4 peak FP16 (Tensor Cores): {fp16_peak / 1e12:5.1f} TFLOPS")   # the Tensor Cores' peak
# the GPU in THIS session: how many SMs does it have?
sms = torch.cuda.get_device_properties(0).multi_processor_count
print(f"This GPU has {sms} SMs")   # show it

In [ ]:
N = 4096   # the matrices are 4096 × 4096
measured = {}   # number format → measured TFLOPS
# dtype is the number format: 32-bit floats, then 16-bit floats
for dtype in [torch.float32, torch.float16]:
    # two random 4096 × 4096 matrices on the GPU, in this format
    a = torch.rand(N, N, device=device, dtype=dtype); b = torch.rand(N, N, device=device, dtype=dtype)
    # average time of one multiply; speed = 2N³ FLOPs ÷ time, ÷ 1e12 gives TFLOPS
    t = timeit(lambda: a @ b, repeats=10)
    measured[dtype] = 2 * N**3 / t / 1e12   # speed = 2N³ FLOPs ÷ time, in TFLOPS
    print(f"{str(dtype):15s}: {t*1e3:7.2f} ms  →  {measured[dtype]:5.1f} TFLOPS measured")   # show the time and the speed
print(f"FP16 is {measured[torch.float16] / measured[torch.float32]:.1f}× faster than FP32 on this GPU")   # compare the two formats

### ✏️ YOUR TURN (A)
What percentage of the T4's computed peak did each precision reach? Why is FP16 so much faster? (Hint: which units does it use?)

*Your answer:*

---
## Part B · Branch Divergence  *(Unit 3.2)*

Below is a real CUDA kernel. Every thread takes one of two expensive paths.
- `mode = 0`: the path is chosen by `(i / 32) % 2`, so all 32 threads in a warp agree.
- `mode = 1`: the path is chosen by `i % 2`, so every warp splits in half.

Same threads, same total work. Only the arrangement differs.

In [ ]:
# A CUDA C kernel, kept in a Python string. In C, comments start with // instead of #.
branchy_src = r'''
// __global__ marks a kernel: code that runs on the GPU and is launched from the CPU
extern "C" __global__
void branchy(float *x, int n, int mode, int work) {   // x = the numbers, n = how many, mode = 0 or 1, work = loop steps
    // this thread's element; spare threads past the end stop at once
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;   // spare threads past the end stop here
    // mode 0: the path depends on (i / 32) % 2, so all 32 threads of a warp agree
    // mode 1: the path depends on i % 2, so neighbouring threads disagree
    int path = (mode == 0) ? ((i / 32) % 2) : (i % 2);
    float v = x[i];   // read this thread's number
    // two equally expensive paths, A and B
    if (path) { for (int k = 0; k < work; k++) v = v * 0.9999f + 0.0001f; }
    else      { for (int k = 0; k < work; k++) v = v * 1.0001f - 0.0001f; }   // (path B, just as long as path A)
    x[i] = v;   // write the result back
}
'''
# CuPy compiles the C code into a GPU kernel
branchy = cp.RawKernel(branchy_src, "branchy")
# 1 << 22 = 2²² ≈ 42 lakh threads, each doing 2,000 loop steps; an array of 1.0s in GPU memory
n, work = 1 << 22, 2000
x = cp.ones(n, dtype=cp.float32)   # an array of 1.0s in GPU memory
# 256 threads per block, and enough blocks to cover n (rounded up)
threads = 256; blocks = (n + threads - 1) // threads
times = {}   # mode → time in ms
for mode, name in [(0, "warps agree   (i/32)%2"), (1, "warps diverge  i%2   ")]:   # run both arrangements
    # launch: kernel((blocks,), (threads per block,), (the kernel's arguments)); cp.int32 is a C "int"
    times[mode] = cuda_time_ms(lambda: branchy((blocks,), (threads,), (x, cp.int32(n), cp.int32(mode), cp.int32(work))))
    print(f"{name}: {times[mode]:7.2f} ms")   # show this arrangement's time
print(f"Divergence cost: {times[1] / times[0]:.2f}× slower")   # how much slower the diverging version was

### ✏️ YOUR TURN (B)
1. Was the diverging version about 2× slower? Explain why using the word *warp*.
2. Change `(i / 32) % 2` to `(i / 16) % 2` and re-run. Predict first: does it diverge?

*Your answers:*

---
## Part C · Hiding Latency  *(Unit 3.3)*

A copy kernel where each thread copies elements in a loop (a "grid-stride loop"). We launch **more and more threads** for the same 256 MB copy.
With few threads, each warp mostly waits for memory. With many, the scheduler always has a ready warp.

In [ ]:
copy_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void copy(const float *x, float *y, int n) {   // copy x into y, n numbers
    // the total number of threads launched
    int stride = gridDim.x * blockDim.x;
    // each thread starts at its own index, copies, then jumps ahead by "stride" until it passes the end
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        y[i] = x[i];   // copy one element
}
'''
copy_k = cp.RawKernel(copy_src, "copy")   # compile the kernel
# 6.7 crore floats = 256 MB to copy; the source and destination both live in GPU memory
n = 1 << 26
xs, ys = cp.random.rand(n, dtype=cp.float32), cp.empty(n, dtype=cp.float32)   # source and destination on the GPU
total_threads, bandwidth = [], []   # the results go into these lists
# launch more and more blocks of 256 threads for the same copy
for blocks in [1, 4, 16, 40, 80, 160, 320, 640, 1280]:
    ms = cuda_time_ms(lambda: copy_k((blocks,), (256,), (xs, ys, cp.int32(n))), repeats=3)   # time the copy with this many blocks
    # bytes moved (read + write, 4 bytes each) ÷ seconds ÷ 1e9 = GB/s
    gbs = 2 * n * 4 / (ms / 1e3) / 1e9
    total_threads.append(blocks * 256); bandwidth.append(gbs)   # keep the results for the chart
    # warps per SM = threads ÷ 32 threads per warp ÷ the number of SMs
    print(f"{blocks*256:>8,d} threads ({blocks*256/32/sms:6.1f} warps per SM): {gbs:6.1f} GB/s")

In [ ]:
plt.figure(figsize=(6, 3.5))                                  # a new chart
plt.semilogx(total_threads, bandwidth, "o-")                  # log scale on the x axis only
plt.xlabel("threads launched"); plt.ylabel("GB/s")            # axis labels
plt.grid(alpha=0.3, which="both")                             # faint grid
plt.title("More warps → memory latency hidden → bandwidth rises, then levels off")   # the chart's title
plt.show()                                                    # draw it

### ✏️ YOUR TURN (C)
Roughly how many warps per SM did it take before the bandwidth levelled off? Connect this to the tea master in Unit 3.3.

*Your answer:*

---
## Part D · Bandwidth: Coalesced vs Strided  *(Unit 3.4)*

First, the best-case bandwidth: copy a big tensor inside GPU memory with PyTorch.

In [ ]:
# 13.4 crore floats = 512 MB; .clone() copies it inside GPU memory: the best case for bandwidth
big = torch.rand(1 << 27, device=device)
t = timeit(lambda: big.clone(), repeats=5)   # time one copy
# bandwidth = bytes read + bytes written ÷ time
best_bw = 2 * big.numel() * 4 / t
print(f"Copy 512 MB inside the GPU: {t*1e3:.1f} ms → {best_bw / 1e9:.0f} GB/s  (a T4's published peak: 320 GB/s)")   # show the bandwidth

Now a kernel where thread i reads element `(i × stride) % n`. With stride 1, neighbouring threads read neighbouring floats (coalesced). With bigger strides, the threads of a warp land in different pieces of memory.

In [ ]:
stride_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void strided(const float *x, float *y, int n, int stride) {   // copy with a gap of "stride" between the reads
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's index
    // thread i reads position i × stride (wrapped round with % n); long long avoids overflow
    if (i < n) y[i] = x[(long long)i * stride % n];
}
'''
strided = cp.RawKernel(stride_src, "strided")   # compile the kernel
# about 3.4 crore floats = 128 MB, one thread per element, 256 threads per block
n = 1 << 25
xs, ys = cp.random.rand(n, dtype=cp.float32), cp.empty(n, dtype=cp.float32)   # source and destination on the GPU
blocks = (n + 255) // 256   # enough blocks of 256 threads to cover n (rounded up)
stride_ms = {}   # stride → time in ms
# stride 1: neighbours read neighbours; stride 32: every thread of a warp lands in a different chunk
for s in [1, 2, 8, 32]:
    stride_ms[s] = cuda_time_ms(lambda: strided((blocks,), (256,), (xs, ys, cp.int32(n), cp.int32(s))))   # time the kernel with this stride
    print(f"stride {s:2d}: {stride_ms[s]:6.2f} ms   useful bandwidth {2 * n * 4 / (stride_ms[s] / 1e3) / 1e9:6.1f} GB/s")   # show the time and the useful bandwidth

### ✏️ YOUR TURN (D)
How many times slower was stride 32 than stride 1? Why doesn't stride 2 hurt as much?

*Your answer:*

---
## Part E · Your Plan vs Reality  *(Unit 3.C)*

The capstone planner predicts lower bounds from FLOPs and bytes, using **this GPU's own best speeds** measured in Parts A and D. Then we measure.

In [ ]:
# plan with this GPU's own best speeds, measured in Parts A and D
PEAK, BW = measured[torch.float32] * 1e12, best_bw
N = 4096   # the matrices are 4096 × 4096
# the layer's 4096 × 4096 weight matrix
W = torch.rand(N, N, device=device)
# the table heading (:16s and :>10s pad to a fixed width so the columns line up)
print(f"{'job':16s} {'plan (ms)':>10s} {'limit':>8s} {'measured':>10s} {'achieved':>14s}")
for B in [1, 64, 1024]:   # three batch sizes
    # a batch of B inputs
    X = torch.rand(B, N, device=device)
    # the layer: FLOPs = 2·B·N·N; bytes = read X, read W, write the output (4 bytes per number)
    flops, nbytes = 2 * B * N * N, (B * N + N * N + B * N) * 4
    # the plan is the bigger of the compute time and the memory time; then measure the reality
    plan_ms = max(flops / PEAK, nbytes / BW) * 1e3
    t = timeit(lambda: X @ W, repeats=10)   # the reality: time the layer
    print(f"matmul B={B:<6d} {plan_ms:10.3f} {'compute' if flops/PEAK > nbytes/BW else 'memory':>8s} {t*1e3:10.3f}   {flops / t / 1e12:6.2f} TFLOPS")   # plan, limit, measured time, achieved speed
    # ReLU on the output: almost no maths, it just reads and writes B·N numbers
    Y = X @ W
    nbytes = 2 * B * N * 4   # ReLU reads and writes B × N numbers of 4 bytes
    t = timeit(lambda: torch.relu(Y), repeats=10)   # time it
    print(f"ReLU   B={B:<6d} {nbytes/BW*1e3:10.3f} {'memory':>8s} {t*1e3:10.3f}   {nbytes / t / 1e9:6.1f} GB/s")   # plan, limit, measured time, achieved bandwidth

### ✏️ YOUR TURN (E) — record your results
| Job | Planned limit | Measured time | Achieved |
|---|---|---|---|
| matmul B=1 | | | |
| matmul B=1024 | | | |
| ReLU B=1024 | | | |

1. Which jobs came closest to their plan? Which were furthest away, and why might tiny jobs miss by a lot? (Hint: launch overhead, Unit 2.5.)
2. Why does the B = 1024 matmul achieve far more TFLOPS than B = 1?

*Your answers:*

---
## 🎓 Wrap-up
- A GPU = many SMs; an SM = schedulers + registers + CUDA cores + Tensor Cores; work runs in warps of 32.
- Divergence costs time; many warps hide memory latency; coalesced access gets the bandwidth.
- Every job is limited by either compute or memory; the bigger lower bound wins.
- **Next — Unit 4: CPU–GPU Teamwork (Lab 4).** You'll allocate, copy, launch your own kernel, and copy back, step by step.